# Kaggle コミットで W&B sweep agent を回すテンプレ

> **初めての人へ:** 画面のどこを押すかを含めた詳しい手順は、リポジトリの `docs/GUIDE.md` §6「Kaggle の使い方」にステップバイステップでまとめてあります。まずそちらを読んでから戻ってくるのがおすすめです。

**注意事項（読んでから実行）**

- このノートは **CPU セッション** で使う。GPU は選択しない（GPU 枠を消費しない）。
- **インターネット接続を有効化** すること（git clone・pip install・wandb 同期に必要）。Settings → Internet を On。
  - インターネット接続の有効化には **電話番号認証** が必要（Add-ons / Settings の "Get phone verified"）。
- W&B の **API キーは直書きしない**。Add-ons → Secrets に `WANDB_API_KEY` として登録し（チェックボックスを On にしてノートに紐付ける）、下のセルで読み込む。
- 最後のセルの `<N>` と `<entity/project/sweep_id>` を実際の値に書き換えてから実行する（山かっこごと消す）。
- 使い方: 右上の Save Version → 「**Save & Run All（Commit）**」で放置実行する。セルを手で1個ずつ実行する方式だと、ブラウザを閉じたとき止まる。

In [ ]:
# 1. リポジトリを clone
# 【なぜ】Kaggle が用意するマシンはまっさらで、私たちのコードが入っていない。
#         まず GitHub からコード一式をこのマシンにダウンロードする。
# 【成功すると】SB3-Team フォルダができて src/train.py などが使える状態になり、
#              %cd でそのフォルダに移動する。
# （このリポジトリは Public なのでトークン不要。ここで失敗する場合は
#   Internet 設定が Off か、電話番号認証が未了の可能性が高い）
!git clone https://github.com/shironaganegi/SB3-Team.git
%cd SB3-Team

In [ ]:
# 2. 依存ライブラリをインストール
# 【なぜ】学習コードは PyTorch・gymnasium・sb3-contrib などに依存しているが、
#         まっさらなマシンには入っていない（または版が合わない）ため。
# 【成功すると】src/train.py が動かせる環境が整う。数分かかるので気長に待つ。
!pip install -r requirements.txt

In [ ]:
# 3. W&B にログイン
# 【なぜ】学習結果を W&B のダッシュボードに送るには、このマシンが
#         「あなたのアカウントである」と名乗る必要がある。無人実行なので
#         パスワード手入力の代わりに API キーで自動ログインする。
# 【どこから読む】キーは Add-ons → Secrets に WANDB_API_KEY として登録した金庫
#         から取り出す（コードに直書きすると公開時に漏れるため）。
# 【成功すると】以降の学習結果がチームの W&B ページに自動で送られるようになる。
# （Secret not found エラーが出たら、Secrets の名前の綴りとチェックボックス On を確認）
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
!wandb login --relogin $WANDB_API_KEY

In [ ]:
# 4. sweep agent を実行（このノートの仕事の本体）
# 【なぜ】sweep は W&B 側にある「試すべきハイパラ設定を配る司令塔」。
#         wandb agent は司令塔に「次の設定をください」と問い合わせ、
#         もらった設定で src/train.py を1本学習 → 終わったらまた次をもらう、
#         を <N> 回繰り返す。全員が同じ sweep ID を使えば重複なく手分けできる。
# 【成功すると】W&B の sweep ページに run が1本ずつ増えていく。
# 【書き換える場所】山かっこ <> ごと消して実際の値にする。
#   <N>  = このセッションで回す本数。1本数時間かかるので、セッション上限
#          （約12時間）に収まるよう 3〜5 程度が目安。
#   <entity/project/sweep_id> = sweep 作成時に表示されるID
#          （例: taro/bipedal-timetrial/abc123xy）。作った人に聞く。
#   sweep は誰か1人が手元で一度だけ作成する: `wandb sweep sweeps/classic_sweep.yaml`
!wandb agent --count <N> <entity/project/sweep_id>